# Assignment 2 Diagnose and improve a generator
**GAN track · Notebook v5 · Chonnam National University · Fall 2026**

Choose **VAE OR GAN**, not both. 두 트랙 중 **하나만** 선택하세요.
Released 15 October 2026 · Due 4 November 2026, 23:59 KST · 12 points · 10% of course grade.

## Your task / 과제의 목표
Inspect a baseline generator, identify a visible limitation, change **one** setting, and decide whether the generated distribution improved. Improvement is not guaranteed and is not required for full marks.

Baseline의 실제 생성 결과에서 문제를 찾고, 설정 하나를 바꾼 뒤 개선 여부를 판단합니다. **좋아진 결과 자체가 아니라 근거 있는 판단을 평가합니다.**

This is a small GAN generating 2D points, not images. Real data have eight clusters. Both runs use the SAME non-saturating loss; only G learning rate changes. / 이 GAN은 이미지가 아닌 2D point를 생성합니다. 두 run 모두 non-saturating loss를 사용하고 G learning rate만 바꿉니다.

**Active work: about 3–4 hours, up to 5 if debugging is needed.** CPU is sufficient. Only two training runs are required. Expect a few minutes for the pair on Colab; actual time depends on the machine and data download.

직접 작업 시간은 약 3–4시간이며, 오류 해결이 필요하면 5시간까지 예상합니다. Baseline 1회와 변경 실험 1회만 수행합니다. 추가 seed 실험이나 hyperparameter 탐색은 요구하지 않습니다.

1. Save a copy in Drive and run Setup. / Drive에 사본을 저장하고 Setup을 실행합니다.
2. Run the baseline and inspect generated samples. / Baseline을 실행하고 생성 결과를 관찰합니다.
3. Write your diagnosis and commit ONE change before seeing its result. / 변경 실험을 보기 전에 관찰·설정·예측·판단 기준을 작성합니다.
4. Complete three short code blanks or use the provided code, then train the intervention. / 코드 세 칸을 완성하거나 제공 코드로 전환한 뒤 변경 실험을 실행합니다.
5. Compare samples and metrics, export the receipt, and write a short report. / 생성 결과와 지표를 비교하고 결과 기록 PDF와 짧은 보고서를 작성합니다.

Submit the executed notebook and ONE PDF containing your 1–2 page analysis followed by the two-page receipt. See `A2_Student_Assignment_v6.pdf` for the rubric. English or Korean answers are accepted.

실행한 노트북과 PDF 하나를 제출하세요. PDF는 분석 본문 1–2쪽 뒤에 자동 생성한 결과 기록 2쪽을 붙입니다. 영어·한국어 답안 모두 가능합니다.

## 0 Setup / 준비
Run these supplied cells without editing. No account, Drive mount, GPU, or paid service is needed.

준비 셀은 수정하지 않고 실행하세요. 계정 연결이나 Drive mount가 필요하지 않습니다. 코드의 제공 함수는 공개된 실행 지원 도구이며, 숨겨진 정답이 아닙니다. Reference 사용은 감점하지 않고 결과 기록에 표시합니다. 분석 답안은 직접 작성해야 합니다.

In [ ]:
#@title Setup and workflow helpers / 준비와 실행 관리 {display-mode:"form"}
import copy, gzip, hashlib, json, math, os, platform, struct, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, FileLink

torch.set_num_threads(2)
VERSION = 'A2 redesign v6 / notebook v5'
SEED = 0
BASE, VARIANT, PLAN, AUDIT = None, None, None, None
CODE_STATUS = {}
OUT = Path('a2_outputs'); OUT.mkdir(exist_ok=True)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:16]

def array_id(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

def code_id():
    # Detect ordinary edits after a run; this is not an anti-tampering system.
    return fingerprint({name: (globals()[name].__code__.co_code.hex(), repr(globals()[name].__code__.co_consts)) for name in CODE_NAMES})

def check_code(number, use_reference=False):
    name = CODE_NAMES[number - 1]
    if use_reference:
        globals()[name] = REFERENCES[number - 1]
    try:
        for args in TESTS[number - 1]:
            expected = REFERENCES[number - 1](*copy.deepcopy(args))
            actual = globals()[name](*copy.deepcopy(args))
            if actual is None:
                raise ValueError('빈칸이 남아 있습니다 / the code blank is still None')
            a, b = torch.as_tensor(actual), torch.as_tensor(expected)
            assert a.shape == b.shape and torch.allclose(a.float(), b.float(), atol=1e-6, rtol=1e-5)
            if b.requires_grad:
                assert a.requires_grad, 'gradient 연결을 유지하세요 / preserve autograd'
        mode = 'reference' if globals()[name] is REFERENCES[number - 1] else 'student'
        CODE_STATUS[name] = mode
        print(f'Code {number}: PASS ({mode})')
        return True
    except Exception as exc:
        CODE_STATUS[name] = 'pending'
        print(f'Code {number}: NOT READY. {type(exc).__name__}: {exc}')
        print('식과 tensor shape을 확인하세요. 필요하면 check_code(number, use_reference=True)를 사용하세요. 감점은 없습니다.')
        return False

def record_plan(student_id, diagnosis, candidate, prediction, success_rule):
    global PLAN, VARIANT, AUDIT
    if BASE is None:
        print('먼저 baseline을 실행하세요 / Run the baseline first.'); return
    values = [student_id, diagnosis, prediction, success_rule]
    if any(not isinstance(x, str) or not x.strip() for x in values):
        print('NOT READY: 학번, baseline 관찰, 예측, 판단 기준을 작성하세요.'); return
    if candidate not in OPTIONS:
        print('NOT READY: 제공된 두 설정 중 하나를 선택하세요.'); return
    draft = dict(student_id=student_id.strip(), diagnosis=diagnosis.strip(), candidate=float(candidate),
                 prediction=prediction.strip(), success_rule=success_rule.strip(), baseline_id=BASE['id'])
    if PLAN != draft:
        PLAN = draft; VARIANT = None; AUDIT = None
    print('PLAN SAVED. 변경 실험을 실행하세요 / Run your chosen intervention.')

def run_baseline():
    global BASE, VARIANT, PLAN, AUDIT
    cfg = config(BASE_VALUE)
    if BASE is not None and BASE['config'] == cfg:
        print('기존 baseline 재사용 / Reusing baseline.'); return
    BASE = train(BASE_VALUE, student=False)
    PLAN = VARIANT = AUDIT = None

def run_variant():
    global VARIANT, AUDIT
    if PLAN is None or BASE is None:
        print('NOT READY: baseline을 보고 변경 계획을 먼저 작성하세요.'); return
    if BASE['config'] != config(BASE_VALUE) or PLAN['baseline_id'] != BASE['id']:
        print('NOT READY: 설정이 바뀌었습니다. baseline부터 다시 실행하세요.'); return
    if not all([check_code(i) for i in [1, 2, 3]]):
        print('NOT READY: 코드 세 칸을 완성하거나 제공 코드로 전환하세요.'); return
    key = fingerprint(PLAN)
    if VARIANT is not None and VARIANT['plan_id'] == key and VARIANT['code_id'] == code_id():
        print('기존 변경 실험 재사용 / Reusing intervention run.'); return
    VARIANT = train(PLAN['candidate'], student=True)
    VARIANT.update(plan_id=key, code_id=code_id())
    AUDIT = None

def ready_reason():
    if BASE is None or PLAN is None or VARIANT is None:
        return 'baseline, plan 또는 intervention 실행이 남아 있습니다.'
    if BASE['config'] != config(BASE_VALUE) or VARIANT['config'] != config(PLAN['candidate']):
        return '설정이 바뀌었습니다. baseline부터 다시 실행하세요.'
    if VARIANT['plan_id'] != fingerprint(PLAN) or PLAN['baseline_id'] != BASE['id']:
        return '변경 계획과 실행 결과가 다릅니다.'
    if VARIANT['code_id'] != code_id() or any(CODE_STATUS.get(n) == 'pending' for n in CODE_NAMES):
        return '코드가 바뀌었습니다. 코드 점검과 변경 실험을 다시 실행하세요.'
    return ''

def make_receipt(ai_use):
    reason = ready_reason()
    if not reason and TRACK == 'VAE' and 'OFFLINE' in DATA_NAME:
        reason = '오프라인 smoke test입니다. 제출 전 MNIST로 두 run을 실행하세요.'
    if not reason and (not isinstance(ai_use, str) or not ai_use.strip()):
        reason = 'AI 사용 내용을 쓰세요. 사용하지 않았다면 none을 적으세요.'
    if not reason and TRACK == 'VAE' and (AUDIT is None or AUDIT['run_id'] != VARIANT['id']):
        reason = '두 grid의 sample 01–16에 대한 visual audit를 작성하세요.'
    if reason:
        print('NOT READY:', reason); return None
    b, v = metrics(BASE), metrics(VARIANT)
    payload = dict(version=VERSION, track=TRACK, plan=PLAN, code=CODE_STATUS.copy(),
                   baseline=dict(config=BASE['config'], metrics=b, sample_id=array_id(BASE['samples'])),
                   intervention=dict(config=VARIANT['config'], metrics=v, sample_id=array_id(VARIANT['samples'])),
                   visual_audit=AUDIT if TRACK == 'VAE' else None, ai_use=ai_use.strip(),
                   environment=dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, device='cpu'))
    fid=fingerprint(payload)
    receipt=dict(status='READY', fingerprint=fid, **payload)
    (OUT/f'A2_{TRACK}_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf8')
    # English labels and numbers in the PDF avoid missing Korean fonts on Colab.
    # The full original Korean plan/AI statement is preserved in the notebook and JSON.
    lines=[f'{VERSION} | {TRACK} | READY',f'Fingerprint: {fid}',
           f'Data: {BASE["config"]["data"]}', f'Changed parameter: {PARAMETER}',
           f'Baseline: {BASE_VALUE}   Intervention: {PLAN["candidate"]}',
           'Only the named parameter changes; both runs restart from the same seed.',
           'Settings: '+json.dumps(BASE['config'],ensure_ascii=True), '',
           'BASELINE', *[f'  {k}: {v}' for k,v in b.items()], '',
           'INTERVENTION', *[f'  {k}: {v}' for k,v in v.items()], '',
           'Code: '+str(CODE_STATUS),
           'Full plan, AI statement and annotations: notebook and receipt JSON.',
           'This receipt is an appendix. Write your own analysis in the report.',
           'Fingerprint checks consistency; it does not prove authorship.']
    import textwrap
    wrapped='\n'.join(y for x in lines for y in (textwrap.wrap(x,95) or ['']))
    pdf_path=OUT/f'A2_{TRACK}_receipt.pdf'
    with PdfPages(pdf_path) as pdf:
        fig=plt.figure(figsize=(8.27,11.69));fig.text(.07,.95,wrapped,va='top',fontfamily='monospace',fontsize=8.5);pdf.savefig(fig);plt.close(fig)
        fig=comparison_figure();pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('READY | fingerprint:',fid)
    print('Baseline:',b); print('Intervention:',v)
    print('분석 보고서 뒤에 PDF를 붙이세요. JSON은 노트북과 함께 보관하면 됩니다.')
    display(FileLink(str(pdf_path)));display(FileLink(str(OUT/f'A2_{TRACK}_receipt.json')))
    return receipt

In [ ]:
#@title GAN model and supplied training loop / model과 학습 코드 {display-mode:"form"}
TRACK='GAN'
PARAMETER='generator learning rate'
BASE_VALUE=0.0003
OPTIONS=(0.001,0.01)
STEPS=5000; BATCH=256; Z_DIM=16
D_LR=0.001; STD=0.05; RADIUS=0.2; MIN_COUNT=50
CENTERS=torch.tensor([[2*math.cos(2*math.pi*k/8),2*math.sin(2*math.pi*k/8)] for k in range(8)])
Z_FIXED=torch.randn(1000,Z_DIM,generator=torch.Generator().manual_seed(1234))
REAL_FIXED=(CENTERS[torch.arange(1000)%8]+STD*torch.randn(1000,2,generator=torch.Generator().manual_seed(654))).numpy()

def make_G():
    return nn.Sequential(nn.Linear(Z_DIM,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,2))
def make_D():
    return nn.Sequential(nn.Linear(2,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,1))
def reference_generator_loss(logits):
    return F.binary_cross_entropy_with_logits(logits,torch.ones_like(logits))
def reference_count_clusters(nearest, valid):
    return np.bincount(nearest[valid],minlength=8)
def reference_off_cluster_share(valid):
    return float(np.mean(~valid))
CODE_NAMES=['generator_loss','count_clusters','off_cluster_share']
REFERENCES=[reference_generator_loss,reference_count_clusters,reference_off_cluster_share]
TESTS=[[(torch.tensor([[-2.],[0.],[2.]],requires_grad=True),),(torch.tensor([[-30.],[30.]],requires_grad=True),)],
       [(np.array([0,0,7,2]),np.array([True,False,True,True])),(np.array([1,5]),np.array([False,False]))],
       [(np.array([True,False,True,False]),),(np.array([False,False]),),(np.array([True,True]),)]]

def config(value):
    return dict(data='synthetic 8-cluster ring',seed=SEED,steps=STEPS,batch=BATCH,z_dim=Z_DIM,
                g_lr=float(value),d_lr=D_LR,std=STD,radius=RADIUS,min_count=MIN_COUNT,eval_samples=1000,
                loss='non-saturating',optimizer='Adam betas=(0.5,0.999)',updates='D1 G1')

def train(value,student=False):
    start=time.time();torch.manual_seed(SEED)
    G,D=make_G(),make_D()
    og=torch.optim.Adam(G.parameters(),lr=value,betas=(.5,.999));od=torch.optim.Adam(D.parameters(),lr=D_LR,betas=(.5,.999))
    snaps={};history=[]
    for step in range(1,STEPS+1):
        real=CENTERS[torch.randint(0,8,(BATCH,))]+STD*torch.randn(BATCH,2)
        z=torch.randn(BATCH,Z_DIM);fake=G(z)
        dl=F.binary_cross_entropy_with_logits(D(real),torch.ones(BATCH,1))+F.binary_cross_entropy_with_logits(D(fake.detach()),torch.zeros(BATCH,1))
        od.zero_grad();dl.backward();od.step()
        gl=(generator_loss if student else reference_generator_loss)(D(fake))
        og.zero_grad();gl.backward();og.step()
        if step%100==0:history.append([step,float(dl.detach()),float(gl.detach())])
        if step in [1000,3000,STEPS]:
            with torch.no_grad():snaps[step]=G(Z_FIXED).numpy()
    samples=snaps[STEPS]
    assert np.isfinite(samples).all(),'Non-finite samples: restart the runtime and check Code 1.'
    result=dict(config=config(value),samples=samples,snapshots=snaps,history=history,seconds=round(time.time()-start,2))
    result['id']=fingerprint([result['config'],array_id(samples)])
    print(f'Completed {STEPS} steps in {result["seconds"]} s. G learning rate = {value}')
    return result

def membership(samples):
    distances=np.linalg.norm(samples[:,None,:]-CENTERS.numpy()[None,:,:],axis=2)
    return distances.argmin(1),distances.min(1)<RADIUS

def metrics(run):
    nearest,valid=membership(run['samples'])
    counts=count_clusters(nearest,valid)
    return dict(covered_clusters=int((counts>=MIN_COUNT).sum()),off_cluster_share=round(float(off_cluster_share(valid)),4),counts=counts.tolist())

def draw_points(ax,samples,title):
    ax.scatter(samples[:,0],samples[:,1],s=4,alpha=.6,color='#315d86')
    for k,c in enumerate(CENTERS.numpy()):
        ax.add_patch(plt.Circle(c,RADIUS,fill=False,color='#af5b33',lw=1));ax.text(c[0]+.13,c[1]+.13,str(k),fontsize=8)
    extent=max(2.5,float(np.abs(samples).max())+.1)
    ax.set(xlim=(-extent,extent),ylim=(-extent,extent),title=title,aspect='equal')

def baseline_figure():
    if BASE is None:return
    fig,axes=plt.subplots(1,2,figsize=(9,4))
    draw_points(axes[0],REAL_FIXED,'Real samples | 8 clusters');draw_points(axes[1],BASE['samples'],'Baseline | generated samples')
    lim=max(abs(x) for a in axes for x in a.get_xlim())
    for a in axes:a.set_xlim(-lim,lim);a.set_ylim(-lim,lim)
    fig.tight_layout();plt.show();plt.close(fig)

def comparison_figure():
    fig,axes=plt.subplots(2,3,figsize=(12,8),gridspec_kw={'height_ratios':[1,0.8]})
    for ax,s,t in zip(axes[0],[REAL_FIXED,BASE['samples'],VARIANT['samples']],['Real samples','Baseline','Intervention']):draw_points(ax,s,t)
    lim=max(abs(x) for a in axes[0] for x in a.get_xlim())
    for a in axes[0]:a.set_xlim(-lim,lim);a.set_ylim(-lim,lim)
    for ax,run,title in zip(axes[1],[None,BASE,VARIANT],['Real counts','Baseline counts','Intervention counts']):
        samples=REAL_FIXED if run is None else run['samples'];near,valid=membership(samples)
        counts=count_clusters(near,valid);ax.bar(range(8),counts,color='#315d86');ax.axhline(MIN_COUNT,color='#af5b33',ls='--',label='coverage threshold')
        ax.set(xticks=range(8),xlabel='Cluster',ylabel='Samples',title=title,ylim=(0,1000));ax.legend(fontsize=7)
    fig.suptitle('Generated distribution | same 1,000 evaluation noise vectors',fontsize=14);fig.tight_layout()
    return fig

def show_comparison():
    reason=ready_reason()
    if reason:print('NOT READY:',reason);return
    fig=comparison_figure();fig.savefig(OUT/'GAN_comparison.png',dpi=140);plt.show();plt.close(fig)
    print('Baseline',metrics(BASE));print('Intervention',metrics(VARIANT))
    fig,axes=plt.subplots(2,3,figsize=(11,7))
    for row,run in enumerate([BASE,VARIANT]):
        for ax,(step,sample) in zip(axes[row],run['snapshots'].items()):draw_points(ax,sample,f'{"Baseline" if row==0 else "Intervention"} | step {step}')
    lim=max(abs(x) for a in axes.flat for x in a.get_xlim())
    for a in axes.flat:a.set_xlim(-lim,lim);a.set_ylim(-lim,lim)
    fig.tight_layout();fig.savefig(OUT/'GAN_progress.png',dpi=120);plt.show();plt.close(fig)

def optional_loss_plot():
    if ready_reason():return
    fig,axes=plt.subplots(1,2,figsize=(9,3))
    for run,label in [(BASE,'Baseline'),(VARIANT,'Intervention')]:
        h=np.array(run['history']);axes[0].plot(h[:,0],h[:,1],label=label);axes[1].plot(h[:,0],h[:,2],label=label)
    for a,title in zip(axes,['D loss','G loss (same non-saturating formula)']):a.set(xlabel='Step',title=title);a.legend()
    fig.tight_layout();plt.show();plt.close(fig)

## 1 Observe the baseline / Baseline 관찰
Run the baseline once. Compare real examples with generated samples before reading any loss curve. Keep the original output even if it differs from another student's output.

Baseline을 먼저 실행하세요. Loss curve보다 실제 생성 결과를 먼저 봅니다. 다른 학생과 결과가 달라도 자신의 결과를 그대로 분석하세요.

Each circle marks a known cluster region; labels 0–7 identify clusters, not digit classes. A convincing point near one center does not show that the generator covers all eight clusters.

원은 알려진 cluster 영역이고 0–7은 cluster 번호입니다. 일부 sample이 그럴듯한 위치에 있다고 해서 모든 cluster를 재현한 것은 아닙니다. 비어 있거나 sample이 지나치게 몰린 위치를 찾으세요.

In [ ]:
#@title Train baseline and inspect samples / Baseline 실행
run_baseline()
baseline_figure()

## 2 Commit one change / 변경 계획
Choose 0.001 or 0.01. Only G learning rate changes. Both runs restart from the same initial weights, seed, training length, data order, and training noise. The intervention is NOT a continuation of the baseline. Evaluation uses the same z vectors.

G learning rate 하나만 변경합니다. 두 run은 같은 초기 weight에서 각각 처음부터 학습하며, data 순서와 training noise 및 평가용 z도 같습니다. Baseline을 이어서 학습하지 않습니다.

G learning rate sets the size of its optimizer updates. D learning rate stays 0.001. Increasing G learning rate may help it adapt relative to D, but a larger value is not guaranteed to improve coverage or off-cluster share.

G learning rate는 optimizer update의 크기에 영향을 줍니다. D learning rate는 0.001로 고정합니다. 값을 높이면 G의 적응 양상이 달라지지만 coverage와 off-cluster share가 함께 좋아진다고 보장할 수는 없습니다.

Write one or two sentences per field. Name an expected benefit AND a possible cost in your success rule. Commit the plan before running the intervention; do not rewrite the prediction after seeing its result.

각 칸에 한두 문장으로 작성하세요. 판단 기준에는 기대하는 개선과 함께 확인할 손해를 적으세요. 변경 실험 결과를 본 뒤 원래 예측을 고치지 말고, 보고서에서 차이를 설명하세요.

In [ ]:
#@title Record the plan before the intervention / 변경 전 계획
student_id = '' #@param {type:"string"}
diagnosis = '' #@param {type:"string"}
new_value = 0.001 #@param [0.001, 0.01] {type:"raw"}
prediction = '' #@param {type:"string"}
success_rule = '' #@param {type:"string"}
record_plan(student_id, diagnosis, new_value, prediction, success_rule)

## 3 Three short code blanks / 코드 세 칸
Complete one expression in each cell. The self-check uses small synthetic inputs, so it does not reveal the experiment outcome. Keep tensor gradients connected where needed.

각 셀에서 식 하나만 채우세요. 작은 입력으로 계산을 점검하며, 실험 결과의 정답을 알려 주는 검사는 아닙니다. 필요한 곳에서는 autograd 연결을 유지하세요.

If stuck, change `check_code(1)` to `check_code(1, use_reference=True)` (use 2 or 3 for the other cells). There is no penalty; explain what the code computes in your PDF. If you introduce a SyntaxError, restore the answer line to `None` before switching.

해결되지 않으면 해당 셀을 제공 코드로 전환하세요. 감점은 없지만 PDF의 코드 설명은 작성해야 합니다. SyntaxError가 있으면 답안 줄을 먼저 `None`으로 되돌리세요.

### Code 1 Non-saturating generator loss
Use F.binary_cross_entropy_with_logits with a real target (ones_like). Average over the batch. D returns logits before sigmoid. / D의 sigmoid 적용 전 logit에 real target을 사용해 batch 평균 BCE를 계산하세요.

In [ ]:
def generator_loss(logits):
    answer = None  # TODO: one expression / 식 하나
    return answer

check_code(1)

### Code 2 Count samples in each cluster
nearest is an integer array in 0..7; valid is a Boolean mask for distance < 0.2. Use np.bincount with minlength=8. / valid인 sample만 cluster별로 세세요. sample이 없는 cluster도 0으로 표시합니다.

In [ ]:
def count_clusters(nearest, valid):
    answer = None  # TODO: one expression / 식 하나
    return answer

check_code(2)

### Code 3 Off-cluster share
Return the fraction of False values in the Boolean valid array. / 어느 cluster 영역에도 포함되지 않는 sample의 비율을 계산하세요.

In [ ]:
def off_cluster_share(valid):
    answer = None  # TODO: one expression / 식 하나
    return answer

check_code(3)

In [ ]:
#@title Train your ONE intervention / 변경 실험 한 번 실행
run_variant()

## 4 Judge the generated results / 생성 결과 평가
Use the sample comparison first. Then use the table to support or revise your visual impression. You can conclude improved, mixed, unchanged, or worse. Do not run both options to choose the prettier result.

그림에서 먼저 관찰하고 수치로 확인하세요. 개선·장단점이 섞임·차이 없음·악화 중 어느 결론도 가능합니다. 두 선택지를 모두 실행해 보기 좋은 결과만 고르지 마세요.

**Covered clusters:** among 1,000 generated samples, a cluster is counted if at least 50 lie within distance 0.2 of its known center. **Off-cluster share:** fraction outside all eight regions. A low off-cluster share can coexist with missing modes. The bar charts reveal uneven allocation even when coverage is high. These are toy-data measures, not universal GAN quality metrics. Figures use common axes; no points are intentionally clipped.

Coverage는 1,000개 중 50개 이상이 중심에서 거리 0.2 이내에 있는 cluster의 수입니다. Off-cluster share는 모든 cluster 영역 밖에 있는 sample 비율입니다. 두 지표를 함께 보고 cluster별 개수도 확인하세요. 8개 cluster를 모두 covered로 판정해도 비율이 균등하다는 뜻은 아닙니다.

In [ ]:
#@title Generated samples and supporting measurements / 생성 결과와 지표
show_comparison()

### Optional training curves / 선택 확인
Use these only if they help explain an observation. You do not need to explain every bend, derive logit gradients, or interpret reference loss constants. Curves alone do not decide generation quality.

관찰한 결과를 설명하는 데 필요할 때만 확인하세요. 모든 굴곡을 설명하거나 logit gradient를 유도할 필요는 없습니다.

In [ ]:
#@title Optional curves / 필요한 경우에만 확인
show_optional_curves = False #@param {type:"boolean"}
if show_optional_curves:
    optional_loss_plot()

## 5 Your report / 분석 보고서
Write about 1–2 pages. The saved plan remains in the notebook; summarize it rather than copying all fields or tables. Include three short code explanations, one sentence per blank.

본문은 1–2쪽 정도로 작성하세요. 계획과 표를 전부 옮겨 적지 말고 필요한 내용만 인용합니다. 코드 칸별 계산 내용과 사용 목적도 한 문장씩 설명하세요.

**Q1 Baseline diagnosis (2 points).** Identify one concrete generation problem and support it with two observations. Name cluster IDs or a visible off-cluster region. / 생성 결과의 문제 하나를 정하고 cluster 번호나 sample 위치를 근거로 두 가지 관찰을 제시하세요.

**Q2 One controlled change (2 points).** Explain your choice and original prediction. Name what stayed fixed and why the comparison is fair. State one possible cost you planned to check. / 설정을 선택한 이유와 원래 예측을 설명하세요. 고정한 조건과 그 이유, 함께 확인할 손해를 적으세요.

**Q3 Did generation improve (4 points).** Give a verdict using one direct visual comparison, two relevant numeric comparisons, and one limitation or remaining problem. Distinguish observations from proposed explanations. Explain whether your prediction held. A well-supported negative result earns full credit. / 그림 비교 하나와 수치 비교 두 가지를 근거로 결론을 쓰세요. 남은 문제 또는 판단의 한계 하나를 제시하고, 관찰 사실과 원인 추정을 구분하세요. 예측과 달라도 타당한 분석이면 만점을 받을 수 있습니다.

**Q4 Next experiment and project bridge (2 points).** Propose ONE follow-up that tests an unresolved explanation, with a result that would change your conclusion. Then name a possible final-project dataset/task and a matching sample-quality or coverage check. No extra training or final-project commitment is required. / 아직 확인하지 못한 설명을 검증할 다음 실험 하나와 결론을 바꿀 결과를 적으세요. 이어서 기말 프로젝트 후보 data 또는 task 하나와 그에 맞는 생성 결과 점검 방법을 제시하세요. 추가 실행이나 주제 확정은 요구하지 않습니다.

**Code and reproducibility (2 points).** Three blanks: 0.25 execution + 0.25 correct explanation each; 0.5 for a READY receipt matching the submitted notebook. Reference code has no penalty. A numeric typo is penalized in the affected analysis item only, not again in reproducibility.

코드 세 칸은 각각 실행 0.25점과 설명 0.25점입니다. READY 결과 기록과 제출 노트북이 일치하면 0.5점입니다. 제공 코드 사용은 감점하지 않습니다. 숫자 오타를 두 문항에서 중복 감점하지 않습니다.

## 6 Export the analysis receipt / 결과 기록 저장
Write your AI-use statement (`none` if unused), then run this cell. It produces a two-page PDF appendix and a JSON record in `a2_outputs`. The PDF contains settings, metrics, a fingerprint, and the sample comparison. Your original plan and AI statement remain in the notebook and JSON, including Korean text.

AI 사용 내용과 용도를 적으세요. 사용하지 않았으면 `none`이라고 적습니다. 생성된 결과 기록 PDF 2쪽을 분석 본문 뒤에 붙여 PDF 하나로 제출하세요. JSON은 보관용이며 별도 제출할 필요가 없습니다. Colab의 왼쪽 Files에서 파일을 다운로드할 수 있습니다.

Before submission, use Runtime > Run all once; cached runs are reused unless settings or code changed. Review the regenerated receipt. Save the notebook with outputs. If restarting the runtime, both runs execute again from their seeds. If results change, update the analysis using the new observations while keeping the original prediction.

제출 전에 Runtime > Run all을 실행하고 결과 기록을 다시 확인하세요. 설정·코드가 같으면 기존 run을 재사용합니다. 런타임을 재시작했다면 두 run을 다시 학습합니다. 결과가 달라지면 분석을 갱신하되 원래 예측은 유지하세요. READY는 제출 자료가 갖춰졌다는 뜻이며 분석의 정확성을 보장하지는 않습니다.

In [ ]:
#@title Export receipt / 결과 기록 PDF 생성
ai_use = '' #@param {type:"string"}
receipt = make_receipt(ai_use)